# Docling vs MinerU: Q&A with Parsed Data

In [3]:
from IPython.display import display, Markdown
from pathlib import Path
import subprocess
import torch
from chonkie import NeuralChunker

In [4]:
pdf_path = "data/pdf/arxiv/2403.20331v2.pdf"
print("PDF exists:", Path(pdf_path).exists())
print("PDF path:", pdf_path)

PDF exists: True
PDF path: data/pdf/arxiv/2403.20331v2.pdf


## Parsed Text

* Same chunking strategy applied to both parsed text
* Same retrieval method
* Compare Q&A

#### Parse Text with Docling and MinerU

In [5]:
# Resolve the PDF path (relative to notebook dir if needed)
try:
    from IPython import get_ipython
    ip = get_ipython()
    nb_path = Path(ip.kernel.shell.user_ns.get("__vsc_ipynb_file__", ""))
    nb_dir = nb_path.parent if nb_path.exists() else Path.cwd()
except Exception:
    nb_dir = Path.cwd()

pdf_path = nb_dir / "data" / "pdf" / "arxiv" / "2403.20331v2.pdf"
print(f"PDF path: {pdf_path}")
print(f"PDF exists: {pdf_path.exists()}")

# ---------- Docling parse ----------
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

docling_result = converter.convert(pdf_path)
docling_markdown = docling_result.document.export_to_markdown()
print(f"\nDocling markdown length: {len(docling_markdown):,} chars")

# ---------- MinerU parse ----------
mineru_output_dir = nb_dir / "data" / "pdf" / "arxiv" / "mineru_output"
mineru_output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "mineru",
    "-p", str(pdf_path),
    "-o", str(mineru_output_dir),
    "-b", "pipeline",
    "-l", "en",
    "-f", "true",
    "-t", "true",
]
print(f"\nRunning: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])

parse_dir = mineru_output_dir / pdf_path.stem / "auto"
md_path = parse_dir / f"{pdf_path.stem}.md"

if md_path.exists():
    mineru_markdown = md_path.read_text(encoding="utf-8")
    print(f"\nMinerU markdown length: {len(mineru_markdown):,} chars")
else:
    mineru_markdown = ""
    print(f"\nMinerU markdown not found at {md_path}")

# Store for later cells
parsed_text = {
    "docling": docling_markdown,
    "mineru": mineru_markdown,
}

PDF path: /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/2403.20331v2.pdf
PDF exists: True


/Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-07-12 22:58:53,050 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-12 22:58:53,056 [RapidOCR] download_file.py:60: File exists and is valid: /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-12 22:58:53,057 [RapidOCR] main.py:63: Using /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-12 22:58:53,068 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-12 22:58:53,070 [RapidOCR] download_file.py:60: File exists and is valid: /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/mo


Docling markdown length: 187,228 chars

Running: mineru -p /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/2403.20331v2.pdf -o /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/mineru_output -b pipeline -l en -f true -t true
Start MinerU FastAPI Service: http://127.0.0.1:51318
API documentation: http://127.0.0.1:51318/docs


MinerU markdown length: 128,868 chars


In [6]:
# Use MPS on Apple Silicon (Mac M5)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize the same NeuralChunker for both parsed texts
chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",
    device_map=device,
    min_characters_per_chunk=10,
)

# Apply the same chunking to both Docling and MinerU parsed text
docling_chunks = chunker.chunk(docling_markdown)
mineru_chunks = chunker.chunk(mineru_markdown)

print(f"Docling chunks: {len(docling_chunks)}")
print(f"MinerU chunks:  {len(mineru_chunks)}")

# Store chunk texts for later cells
chunked_text = {
    "docling": [c.text for c in docling_chunks],
    "mineru": [c.text for c in mineru_chunks],
}

Using device: mps


Device set to use mps
Token indices sequence length is longer than the specified maximum sequence length for this model (10569 > 8192). Running this sequence through the model will result in indexing errors


Docling chunks: 96
MinerU chunks:  55


In [7]:
text_query_lst = ['Is detailed justification required when refining problems during the curation process?',
                    'Is there a strong correlation between OC-Dual accuracy and Dual accuracy?',
                    'Does self-reflection allow models to evaluate their own answers?',
                    'What activity involves using ropes and harnesses on a cliff face?']

In [15]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi
import tiktoken

load_dotenv("../../.env")  # adjust if your .env is elsewhere
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"
MAX_TOKENS = 8191  # OpenAI limit for text-embedding-3-small

_tokenizer = tiktoken.get_encoding("cl100k_base")


def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of texts using OpenAI text-embedding-3-small."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)


def cosine_sim_batch(query_vecs: np.ndarray, chunk_vecs: np.ndarray) -> np.ndarray:
    """Vectorized cosine similarity: (Q, D) x (N, D) -> (Q, N)."""
    query_norms = np.linalg.norm(query_vecs, axis=1, keepdims=True)
    chunk_norms = np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
    dot = query_vecs @ chunk_vecs.T
    return dot / np.maximum(query_norms * chunk_norms.T, 1e-10)


def rrf_fuse(rankings: list[list[int]], k: int = 60) -> dict[int, float]:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] += 1.0 / (k + rank + 1)
    return scores


def truncate_to_token_limit(text: str, max_tokens: int = MAX_TOKENS) -> str:
    """Truncate text to stay within OpenAI's token limit using tiktoken."""
    tokens = _tokenizer.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return _tokenizer.decode(tokens[:max_tokens])


def hybrid_top1(chunks: list[str], queries: list[str]) -> list[tuple[str, float]]:
    """Return the top-1 chunk for each query using semantic + BM25 hybrid search."""
    # Build BM25 index on full chunks
    tokenized_corpus = [text.lower().split() for text in chunks]
    bm25 = BM25Okapi(tokenized_corpus)

    # Embed queries in batch
    query_embeddings = embed_texts(queries)

    # Embed chunks in batch after truncation
    truncated_chunks = [truncate_to_token_limit(chunk) for chunk in chunks]
    chunk_embeddings = embed_texts(truncated_chunks)

    # All cosine similarities at once: (Q, N)
    sem_scores_all = cosine_sim_batch(query_embeddings, chunk_embeddings)

    top1_results = []
    for q_idx, query in enumerate(queries):
        sem_scores = sem_scores_all[q_idx]
        bm25_scores = bm25.get_scores(query.lower().split())

        sem_ranking = np.argsort(-sem_scores).tolist()
        kw_ranking = np.argsort(-bm25_scores).tolist()

        fused = rrf_fuse([sem_ranking, kw_ranking])
        top_idx = max(fused, key=lambda i: fused[i])

        top1_results.append((chunks[top_idx], fused[top_idx]))

    return top1_results


# Run retrieval on both Docling and MinerU chunks
docling_top1 = hybrid_top1(chunked_text["docling"], text_query_lst)
mineru_top1 = hybrid_top1(chunked_text["mineru"], text_query_lst)

# Build comparison table
comparison_rows = []
for q, (d_chunk, d_score), (m_chunk, m_score) in zip(text_query_lst, docling_top1, mineru_top1):
    comparison_rows.append({
        "query": q,
        "top1_retrieved_chunk_docling": d_chunk[:500] + ("..." if len(d_chunk) > 500 else ""),
        "top1_retrieved_chunk_mineru": m_chunk[:500] + ("..." if len(m_chunk) > 500 else ""),
    })

df = pd.DataFrame(comparison_rows)
pd.set_option("display.max_colwidth", None)
display(df)


,query,top1_retrieved_chunk_docling,top1_retrieved_chunk_mineru
0,Is detailed justification required when refining problems during the curation process?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not verify the efficacy of either the Option or Instruction approaches. This result reveals that the evaluation using MMMU fails to capture important findings of the effectiveness of these prompting approaches for UPD. Specifically, for expert-level problems, LMMs do not have accurate answers due to the lack of capability. Therefore, even if they choose an incorrect option when encountering...","\n\nClarification of the Semantics of the Options. We clarify the meaning of the options. Specifically, some questions in #6: Attribute Comparison have “Can’t judge”. “Can’t judge” means that “I can’t judge from the image since the image does not have enough information”. However, “Can’t judge” might be interpreted as “Since the given options are incorrect, can’t judge.” Therefore, we changed the option of “Can’t judge” to “Can’t judge from the image due to the lack of image information” to reduce..."
1,Is there a strong correlation between OC-Dual accuracy and Dual accuracy?,Figure A: Relationship between OC-Dual accuracy and Dual accuracy.\n\n<!-- image -->,"\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and additional-instruction setting. The “Orig” (Original Standard) value is the upper bound of Dual accuracy. The results show that the difference between each Dual accuracy and the Original Standard is clear and most open-source LMMs have significantly low scores."
2,Does self-reflection allow models to evaluate their own answers?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not verify the efficacy of either the Option or Instruction approaches. This result reveals that the evaluation using MMMU fails to capture important findings of the effectiveness of these prompting approaches for UPD. Specifically, for expert-level problems, LMMs do not have accurate answers due to the lack of capability. Therefore, even if they choose an incorrect option when encountering...","ail\n\n## C.1 Experimental Setup\n\nComputing Infrastructures. We conduct all our evaluations of open-source models on a single NVIDIA A100 (80GB) GPU.\n\nHyperParameters of LMM Inference. We set a temperature to 0 for all models during inference.\n\n## C.2 Detail of LLM-driven Methods\n\nIn this section, we explain the details of the LLMdriven approaches in Sec. 5.3.\n\nChain of Thought (CoT) Prompting. In this experiment, we investigate whether a widely used Zero-shot CoT (Kojima et al., 2022) is effectiv..."
3,What activity involves using ropes and harnesses on a cliff face?,"\n\nQ. What's the function of the demonstrated object?\n\n## A. running\n\n- B. Play football\n- C. Play basketball\n\n## GPT-4o\n\nNone of the provided options are correct . The object demonstrated in the image is a tennis court, which is used for playing tennis .\n\n## Gemini1.5Pro\n\nThe function of the demonstrated object is to **play tennis**. Although none of the provided options are correct , the image clearly shows a tennis court.\n\n## LLaVA-NeXT-34B\n\nThe object in the image is a tennis racket, which is...","\n\nMarah Abdin, Sam Ade Jacobs, Ammar Ahmad Awan, Jyoti Aneja, Ahmed Awadallah, Hany Awadalla, Nguyen Bach, Amit Bahree, Arash Bakhtiari, Harkirat Behl, et al. 2024. Phi-3 technical report: A highly capable language model locally on your phone. arXiv preprint arXiv:2404.14219.\n\nJosh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Florencia Leoni Aleman, Diogo Almeida, Janko Altenschmidt, Sam Altman, Shyamal Anadkat, et al. 2023. Gpt-4 technical report. arXiv preprint arXiv:2303.0..."


In [16]:
df.to_excel("docling_vs_mineru_qa_comparison.xlsx", index=False)